In [2]:
!pip install bertviz

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.6/157.6 kB 220.0 kB/s eta 0:00:0000:0100:01


In [3]:
import torch
from torch import nn
import math
from bertviz.transformers_neuron_view import BertModel, BertConfig
from transformers import BertTokenizer

In [7]:
max_length = 256
model_name = 'bert-base-uncased'

In [9]:
tokenizer = BertTokenizer.from_pretrained(model_name)

In [10]:
config = BertConfig.from_pretrained(model_name, output_attentions=True, 
                                    output_hidden_states=True, 
                                    return_dict=True)
config.max_position_embeddings = max_length

model = BertModel(config).from_pretrained(model_name)
model = model.eval()

In [11]:
model.config

{
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "finetuning_task": null,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "num_labels": 2,
  "output_attentions": true,
  "output_hidden_states": false,
  "pad_token_id": 0,
  "torchscript": false,
  "type_vocab_size": 2,
  "vocab_size": 30522
}

In [13]:
att_head_size = int(model.config.hidden_size/model.config.num_attention_heads)
att_head_size

64

In [14]:
model.encoder.layer[0]

BertLayer(
  (attention): BertAttention(
    (self): BertSelfAttention(
      (query): Linear(in_features=768, out_features=768, bias=True)
      (key): Linear(in_features=768, out_features=768, bias=True)
      (value): Linear(in_features=768, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (output): BertSelfOutput(
      (dense): Linear(in_features=768, out_features=768, bias=True)
      (LayerNorm): BertLayerNorm()
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (intermediate): BertIntermediate(
    (dense): Linear(in_features=768, out_features=3072, bias=True)
  )
  (output): BertOutput(
    (dense): Linear(in_features=3072, out_features=768, bias=True)
    (LayerNorm): BertLayerNorm()
    (dropout): Dropout(p=0.1, inplace=False)
  )
)

In [15]:
model.encoder.layer[0].attention

BertAttention(
  (self): BertSelfAttention(
    (query): Linear(in_features=768, out_features=768, bias=True)
    (key): Linear(in_features=768, out_features=768, bias=True)
    (value): Linear(in_features=768, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (output): BertSelfOutput(
    (dense): Linear(in_features=768, out_features=768, bias=True)
    (LayerNorm): BertLayerNorm()
    (dropout): Dropout(p=0.1, inplace=False)
  )
)

In [16]:
model.encoder.layer[0].attention.self

BertSelfAttention(
  (query): Linear(in_features=768, out_features=768, bias=True)
  (key): Linear(in_features=768, out_features=768, bias=True)
  (value): Linear(in_features=768, out_features=768, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
)

In [17]:
model.encoder.layer[0].attention.self.query

Linear(in_features=768, out_features=768, bias=True)

In [18]:
# 如何体现多头的机制
model.encoder.layer[0].attention.self.query.weight

Parameter containing:
tensor([[-0.0164,  0.0261, -0.0263,  ...,  0.0154,  0.0768,  0.0548],
        [-0.0326,  0.0346, -0.0423,  ..., -0.0527,  0.1393,  0.0078],
        [ 0.0105,  0.0334,  0.0109,  ..., -0.0279,  0.0258, -0.0468],
        ...,
        [-0.0085,  0.0514,  0.0555,  ...,  0.0282,  0.0543, -0.0541],
        [-0.0198,  0.0944,  0.0617,  ..., -0.1042,  0.0601,  0.0470],
        [ 0.0015, -0.0952,  0.0099,  ..., -0.0191, -0.0508, -0.0085]],
       requires_grad=True)

In [19]:
model.encoder.layer[0].attention.self.query.weight.shape

torch.Size([768, 768])

In [20]:
model.encoder.layer[0].attention.self.query.weight.T

tensor([[-0.0164, -0.0326,  0.0105,  ..., -0.0085, -0.0198,  0.0015],
        [ 0.0261,  0.0346,  0.0334,  ...,  0.0514,  0.0944, -0.0952],
        [-0.0263, -0.0423,  0.0109,  ...,  0.0555,  0.0617,  0.0099],
        ...,
        [ 0.0154, -0.0527, -0.0279,  ...,  0.0282, -0.1042, -0.0191],
        [ 0.0768,  0.1393,  0.0258,  ...,  0.0543,  0.0601, -0.0508],
        [ 0.0548,  0.0078, -0.0468,  ..., -0.0541,  0.0470, -0.0085]],
       grad_fn=<PermuteBackward0>)

In [23]:
# 第一头
model.encoder.layer[0].attention.self.query.weight[:, :64]

tensor([[-0.0164,  0.0261, -0.0263,  ..., -0.0545,  0.0607,  0.0398],
        [-0.0326,  0.0346, -0.0423,  ...,  0.0586, -0.0306, -0.0412],
        [ 0.0105,  0.0334,  0.0109,  ..., -0.0573, -0.0118, -0.0147],
        ...,
        [-0.0085,  0.0514,  0.0555,  ..., -0.0607,  0.0823,  0.0531],
        [-0.0198,  0.0944,  0.0617,  ...,  0.0934, -0.0044, -0.0301],
        [ 0.0015, -0.0952,  0.0099,  ...,  0.0719,  0.0701,  0.0175]],
       grad_fn=<SliceBackward0>)

In [25]:
# 第二头 以此类推
model.encoder.layer[0].attention.self.query.weight[:, 64:128]

tensor([[-0.0311,  0.0519, -0.0027,  ..., -0.0243,  0.0368, -0.0665],
        [-0.0533, -0.0010, -0.0144,  ..., -0.0041,  0.0334,  0.0111],
        [-0.0881, -0.0265,  0.0096,  ...,  0.0037, -0.0163, -0.0429],
        ...,
        [ 0.0842, -0.0022,  0.0176,  ...,  0.0160, -0.0611,  0.0042],
        [-0.0281,  0.0159,  0.0187,  ..., -0.0172, -0.0544,  0.0810],
        [ 0.0362,  0.0182, -0.0246,  ..., -0.0236,  0.0365,  0.0279]],
       grad_fn=<SliceBackward0>)

In [28]:
from sklearn.datasets import fetch_20newsgroups
newsgroups_train = fetch_20newsgroups(subset='train')
inputs_tests = tokenizer(newsgroups_train['data'][:1], 
                         truncation=True, padding=True,
                         return_tensors='pt')

In [29]:
inputs_tests

{'input_ids': tensor([[  101,  2013,  1024,  3393,  2099,  2595,  3367,  1030, 11333,  2213,
          1012,  8529,  2094,  1012,  3968,  2226,  1006,  2073,  1005,  1055,
          2026,  2518,  1007,  3395,  1024,  2054,  2482,  2003,  2023,   999,
          1029,  1050,  3372,  2361,  1011, 14739,  1011,  3677,  1024, 10958,
          2278,  2509,  1012, 11333,  2213,  1012,  8529,  2094,  1012,  3968,
          2226,  3029,  1024,  2118,  1997,  5374,  1010,  2267,  2380,  3210,
          1024,  2321,  1045,  2001,  6603,  2065,  3087,  2041,  2045,  2071,
          4372,  7138,  2368,  2033,  2006,  2023,  2482,  1045,  2387,  1996,
          2060,  2154,  1012,  2009,  2001,  1037,  1016,  1011,  2341,  2998,
          2482,  1010,  2246,  2000,  2022,  2013,  1996,  2397, 20341,  1013,
          2220, 17549,  1012,  2009,  2001,  2170,  1037,  5318,  4115,  1012,
          1996,  4303,  2020,  2428,  2235,  1012,  1999,  2804,  1010,  1996,
          2392, 21519,  2001,  3584,  

In [31]:
inputs_tests['input_ids'].shape

torch.Size([1, 201])

In [33]:
inputs_tests.keys()

dict_keys(['input_ids', 'token_type_ids', 'attention_mask'])

In [32]:
model_output = model(**inputs_tests)

In [ ]:
- last_hidden_state (batch_size, sequence_length, hidden_size) : last hidden state which is outputted from the last BertLayer
- pooler_output (batch_size, hidden_size) : output of the Pooler layer
- hidden_states (batch_size, sequence_length, hidden_size): hidden-states of the model at the output of each BertLayer plus the initial embedding
- attentions (batch_size, num_heads, sequence_length, sequence_length): one for each BertLayer. Attentions weights after the attention SoftMax

In [34]:
len(model_output)

3

In [39]:
# 最后一层是attention
model_output[-1]

({'attn': tensor([[[[5.3477e-03, 1.0850e-02, 5.1914e-03,  ..., 3.9360e-03,
             3.6427e-03, 1.4398e-02],
            [8.5758e-03, 4.0597e-03, 1.2523e-02,  ..., 4.5496e-03,
             4.1374e-03, 7.1238e-03],
            [5.0512e-03, 4.2847e-03, 4.5569e-03,  ..., 4.2947e-03,
             4.5143e-03, 3.1244e-03],
            ...,
            [1.0410e-03, 2.3348e-03, 5.5335e-03,  ..., 1.2235e-03,
             1.7816e-03, 1.1450e-03],
            [1.0483e-03, 2.2994e-03, 5.7017e-03,  ..., 1.2290e-03,
             1.6825e-03, 6.6760e-04],
            [2.1551e-03, 5.6242e-03, 6.2811e-03,  ..., 4.5159e-03,
             4.7504e-03, 1.4801e-03]],
  
           [[1.8761e-02, 8.6471e-03, 1.3796e-03,  ..., 2.3716e-02,
             2.3086e-02, 7.7072e-05],
            [1.4939e-03, 1.2984e-03, 6.6249e-03,  ..., 6.5390e-04,
             7.7479e-04, 1.7725e-03],
            [5.2377e-04, 2.2026e-03, 2.6308e-02,  ..., 2.6647e-04,
             2.9575e-04, 1.5150e-03],
            ...,
         

In [40]:
len(model_output[-1])

12

In [41]:
model_output[-1][0]

{'attn': tensor([[[[5.3477e-03, 1.0850e-02, 5.1914e-03,  ..., 3.9360e-03,
            3.6427e-03, 1.4398e-02],
           [8.5758e-03, 4.0597e-03, 1.2523e-02,  ..., 4.5496e-03,
            4.1374e-03, 7.1238e-03],
           [5.0512e-03, 4.2847e-03, 4.5569e-03,  ..., 4.2947e-03,
            4.5143e-03, 3.1244e-03],
           ...,
           [1.0410e-03, 2.3348e-03, 5.5335e-03,  ..., 1.2235e-03,
            1.7816e-03, 1.1450e-03],
           [1.0483e-03, 2.2994e-03, 5.7017e-03,  ..., 1.2290e-03,
            1.6825e-03, 6.6760e-04],
           [2.1551e-03, 5.6242e-03, 6.2811e-03,  ..., 4.5159e-03,
            4.7504e-03, 1.4801e-03]],
 
          [[1.8761e-02, 8.6471e-03, 1.3796e-03,  ..., 2.3716e-02,
            2.3086e-02, 7.7072e-05],
           [1.4939e-03, 1.2984e-03, 6.6249e-03,  ..., 6.5390e-04,
            7.7479e-04, 1.7725e-03],
           [5.2377e-04, 2.2026e-03, 2.6308e-02,  ..., 2.6647e-04,
            2.9575e-04, 1.5150e-03],
           ...,
           [5.2149e-04, 2.6457

In [42]:
model_output[-1][0].keys()

dict_keys(['attn', 'queries', 'keys'])

In [45]:
model_output[-1][0]['attn'].shape

torch.Size([1, 12, 201, 201])

In [46]:
model_output[-1][0]['queries'].shape

torch.Size([1, 12, 201, 64])

In [47]:
model_output[-1][0]['keys'].shape

torch.Size([1, 12, 201, 64])

In [48]:
model_output[-1][0]['attn'][0,0,:,:]

tensor([[0.0053, 0.0109, 0.0052,  ..., 0.0039, 0.0036, 0.0144],
        [0.0086, 0.0041, 0.0125,  ..., 0.0045, 0.0041, 0.0071],
        [0.0051, 0.0043, 0.0046,  ..., 0.0043, 0.0045, 0.0031],
        ...,
        [0.0010, 0.0023, 0.0055,  ..., 0.0012, 0.0018, 0.0011],
        [0.0010, 0.0023, 0.0057,  ..., 0.0012, 0.0017, 0.0007],
        [0.0022, 0.0056, 0.0063,  ..., 0.0045, 0.0048, 0.0015]],
       grad_fn=<SliceBackward0>)

In [52]:
# 每行都是一个softmax概率化了
model_output[-1][0]['attn'][0,0,:,:].sum(dim=-1)

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 

### 4. from scratch 从0开始

In [53]:
emb_output = model.embeddings(inputs_tests['input_ids'], inputs_tests['token_type_ids'])

In [54]:
emb_output.shape

torch.Size([1, 201, 768])

In [73]:
emb_output

tensor([[[ 0.1686, -0.2858, -0.3261,  ..., -0.0276,  0.0383,  0.1640],
         [-0.1172,  0.6055,  0.0487,  ...,  0.5867,  0.8167,  0.4067],
         [-0.7412,  0.3854, -0.7550,  ...,  0.5425,  0.5629,  0.6106],
         ...,
         [ 0.0679,  0.2560,  0.3443,  ...,  0.5042,  0.4860,  0.3145],
         [ 0.1079,  0.0740,  0.4233,  ...,  0.2864,  0.5379,  0.1220],
         [-0.0594, -0.0563,  0.2673,  ..., -0.7952, -0.0813, -0.6690]]],
       grad_fn=<AddBackward0>)

In [75]:
emb_output[0].shape

torch.Size([201, 768])

In [74]:
emb_output[0]

tensor([[ 0.1686, -0.2858, -0.3261,  ..., -0.0276,  0.0383,  0.1640],
        [-0.1172,  0.6055,  0.0487,  ...,  0.5867,  0.8167,  0.4067],
        [-0.7412,  0.3854, -0.7550,  ...,  0.5425,  0.5629,  0.6106],
        ...,
        [ 0.0679,  0.2560,  0.3443,  ...,  0.5042,  0.4860,  0.3145],
        [ 0.1079,  0.0740,  0.4233,  ...,  0.2864,  0.5379,  0.1220],
        [-0.0594, -0.0563,  0.2673,  ..., -0.7952, -0.0813, -0.6690]],
       grad_fn=<SelectBackward0>)

In [55]:
model.encoder.layer[0]

BertLayer(
  (attention): BertAttention(
    (self): BertSelfAttention(
      (query): Linear(in_features=768, out_features=768, bias=True)
      (key): Linear(in_features=768, out_features=768, bias=True)
      (value): Linear(in_features=768, out_features=768, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (output): BertSelfOutput(
      (dense): Linear(in_features=768, out_features=768, bias=True)
      (LayerNorm): BertLayerNorm()
      (dropout): Dropout(p=0.1, inplace=False)
    )
  )
  (intermediate): BertIntermediate(
    (dense): Linear(in_features=768, out_features=3072, bias=True)
  )
  (output): BertOutput(
    (dense): Linear(in_features=3072, out_features=768, bias=True)
    (LayerNorm): BertLayerNorm()
    (dropout): Dropout(p=0.1, inplace=False)
  )
)

In [77]:
# emb_output[0].shape: 201*768
# query.weight.T.shape: 768*768, query.weight.T[:, :att_head_size]: 768*64
# 201*64
# https://pytorch.org/docs/stable/generated/torch.nn.Linear.html#torch.nn.Linear
# y=xAT+b
Q_first_head_first_layer = emb_output[0] @ model.encoder.layer[0].attention.self.query.weight.T[:, :att_head_size] \
                            + model.encoder.layer[0].attention.self.query.bias[:att_head_size]

In [57]:
Q_first_head_first_layer.shape

torch.Size([201, 64])

In [76]:
model.encoder.layer[0].attention.self.query.bias.shape

torch.Size([768])

In [58]:
# 201*64
K_first_head_first_layer = emb_output[0] @ model.encoder.layer[0].attention.self.key.weight.T[:, :att_head_size] \
                            + model.encoder.layer[0].attention.self.key.bias[:att_head_size]

In [59]:
K_first_head_first_layer.shape

torch.Size([201, 64])

In [60]:
# (201*64)*(64*201) ==> 201*201
attn_scores = torch.nn.Softmax(dim=-1)(Q_first_head_first_layer @ K_first_head_first_layer.T / math.sqrt(att_head_size))

In [61]:
attn_scores

tensor([[0.0053, 0.0109, 0.0052,  ..., 0.0039, 0.0036, 0.0144],
        [0.0086, 0.0041, 0.0125,  ..., 0.0045, 0.0041, 0.0071],
        [0.0051, 0.0043, 0.0046,  ..., 0.0043, 0.0045, 0.0031],
        ...,
        [0.0010, 0.0023, 0.0055,  ..., 0.0012, 0.0018, 0.0011],
        [0.0010, 0.0023, 0.0057,  ..., 0.0012, 0.0017, 0.0007],
        [0.0022, 0.0056, 0.0063,  ..., 0.0045, 0.0048, 0.0015]],
       grad_fn=<SoftmaxBackward0>)

In [62]:
attn_scores.sum(dim=-1)

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000,
        1.0000, 1.0000, 1.0000, 1.0000, 

In [63]:
V_first_head_first_layer = emb_output[0] @ model.encoder.layer[0].attention.self.value.weight.T[:, :att_head_size] \
                            + model.encoder.layer[0].attention.self.value.bias[:att_head_size]

In [66]:
V_first_head_first_layer.shape

torch.Size([201, 64])

In [71]:
V_first_head_first_layer

tensor([[ 2.3786e+00, -9.7945e-02, -2.8436e-01,  ...,  9.6968e-02,
         -1.8537e-01,  2.1132e-01],
        [ 9.8886e-02,  2.2942e-01, -3.9613e-01,  ...,  2.0846e-01,
         -1.1224e-01, -2.1420e-01],
        [-6.3356e-01,  1.1083e+00, -5.2854e-04,  ..., -3.1870e-01,
          7.2633e-02, -1.0239e-01],
        ...,
        [-7.9659e-01, -4.9400e-01, -4.9216e-02,  ..., -3.6446e-01,
          3.1565e-01, -7.1713e-01],
        [-9.2551e-01, -5.0348e-01, -1.0398e-01,  ..., -2.1418e-01,
          1.6604e-01, -5.9637e-01],
        [ 1.2915e-01, -7.5899e-03,  1.8397e-01,  ...,  2.6980e-01,
          1.3651e-01,  1.9180e-01]], grad_fn=<AddBackward0>)

In [64]:
attn_emb = attn_scores @ V_first_head_first_layer

In [65]:
attn_emb.shape

torch.Size([201, 64])

In [70]:
attn_emb

tensor([[-4.5640e-01,  4.6211e-02,  4.3913e-02,  ..., -2.0099e-02,
         -1.2756e-02,  6.4255e-03],
        [-4.5674e-01,  3.4322e-02,  3.2707e-02,  ..., -4.9206e-02,
          1.4975e-02, -3.0628e-02],
        [-4.9474e-01, -2.9539e-04, -7.5372e-04,  ..., -2.0035e-02,
          1.7146e-02, -3.0126e-02],
        ...,
        [-3.7991e-01,  5.2831e-02,  2.2534e-02,  ..., -1.8338e-02,
         -6.9508e-02,  2.1317e-02],
        [-3.8071e-01,  4.0900e-02,  2.8770e-02,  ..., -2.1192e-02,
         -5.2893e-02,  1.9734e-02],
        [-4.7131e-01,  1.0947e-01,  1.1631e-02,  ..., -3.4542e-02,
         -2.3753e-02, -5.0505e-03]], grad_fn=<MmBackward0>)

### 5. all
- bias偏差
- T转置
- weight权重
- @是一个操作符，表示矩阵-向量乘法

In [67]:
Q_first_layer = emb_output[0] @ model.encoder.layer[0].attention.self.query.weight.T \
                            + model.encoder.layer[0].attention.self.query.bias
K_first_layer = emb_output[0] @ model.encoder.layer[0].attention.self.key.weight.T \
                            + model.encoder.layer[0].attention.self.key.bias
V_first_layer = emb_output[0] @ model.encoder.layer[0].attention.self.value.weight.T \
                            + model.encoder.layer[0].attention.self.value.bias

In [72]:
Q_first_layer.shape

torch.Size([201, 768])

In [68]:
scores = torch.nn.Softmax(dim=-1)(Q_first_layer @ K_first_layer.T / math.sqrt(64))

In [69]:
scores @ V_first_layer[:, :64]

tensor([[ 2.3786e+00, -9.7945e-02, -2.8436e-01,  ...,  9.6968e-02,
         -1.8537e-01,  2.1132e-01],
        [-6.2984e-01,  1.1037e+00, -2.0761e-04,  ..., -3.1767e-01,
          7.2197e-02, -1.0126e-01],
        [ 3.7932e-02,  2.9540e-01, -3.6121e-01,  ...,  1.6694e-01,
         -9.9637e-02, -2.0356e-01],
        ...,
        [-7.0623e-01, -3.0113e-01,  9.4959e-02,  ..., -1.7217e-01,
          1.8647e-01, -4.4743e-01],
        [ 6.2046e-02, -4.0626e-02,  1.6757e-01,  ...,  2.2690e-01,
          1.4684e-01,  1.3030e-01],
        [ 2.3748e+00, -9.7801e-02, -2.8357e-01,  ...,  9.7254e-02,
         -1.8481e-01,  2.1127e-01]], grad_fn=<MmBackward0>)